In [1]:
import bw2io as bi
import bw2data as bd
import bw2calc as bc
import scipy

In [2]:
bd.projects.set_current("SEE_LAB")

In [3]:
technosphere = bd.Database("ecoinvent-3.9.1-cutoff")
biosphere="ecoinvent-3.9.1-biosphere"
foreground="Water bottle LCA"
biosphere, foreground = bd.Database(biosphere), bd.Database(foreground)

In [4]:
ef_gwp_key = [m for m in bd.methods if "climate change" in m[1] and "EF" in m[0]].pop()
# ef_gwp_key = set([m for m in bd.methods]).pop()

my_functional_unit, data_objs, _ = bd.prepare_lca_inputs(
    {act: 1 for act in foreground},  # Using the first activity as the functional unit
    method=ef_gwp_key,  # Using the chosen method 
)

In [5]:
ef_gwp_key

('EF v3.1 EN15804',
 'climate change: land use and land use change',
 'global warming potential (GWP100)')

In [6]:
my_functional_unit, data_objs

({61165: 1, 61164: 1},
  <bw_processing.datapackage.Datapackage at 0x1a339bc4050>])

In [7]:
my_lca = bc.LCA(demand=my_functional_unit, data_objs=data_objs)


In [8]:
my_lca.load_lci_data()
my_lca.load_lcia_data()

In [ ]:
A=my_lca.technosphere_matrix
B=my_lca.biosphere_matrix
C=my_lca.characterization_matrix


In [11]:
print(
    A.shape,
    A.nnz,
    '\n',
    B.shape,
    B.nnz,
    '\n',
    C.shape,
    C.nnz,
)

(21240, 21240) 266446 
 (2420, 21240) 407352 
 (2420, 2420) 5


In [12]:
g = B * A
g.shape

(2420, 21240)

In [13]:
L = C*g

In [14]:
Q = L.transpose() * L

In [15]:
Q.shape 

(21240, 21240)

In [16]:
print(
    Q.shape,
    Q.nnz,
)

(21240, 21240) 999459


In [17]:
a, b = Q.shape
(Q.nnz) / (a*b)

0.0022154194197069807

In [18]:
evals, evecs = scipy.sparse.linalg.eigsh(Q)

In [19]:
print(evals)

[8.53386468e-06 1.29974354e+00 3.99072245e+02 1.18607907e+08
 2.42548498e+08 6.64752363e+10]


In [20]:
rows, cols = C.nonzero()
list(zip(rows, cols))

[(1703, 1703), (1704, 1704), (1705, 1705), (1707, 1707), (1754, 1754)]

In [21]:
evals_a, evacs_s = scipy.sparse.linalg.eigs(A)

In [22]:
print(max(evals_a))

(1.676922487531575+0j)


In [23]:
bd.methods

Methods dictionary with 895 objects, including:
	('CML v4.8 2016', 'acidification', 'acidification (incl. fate, average Europe total, A&B)')
	('CML v4.8 2016', 'climate change', 'global warming potential (GWP100)')
	('CML v4.8 2016', 'ecotoxicity: freshwater', 'freshwater aquatic ecotoxicity (FAETP inf)')
	('CML v4.8 2016', 'ecotoxicity: marine', 'marine aquatic ecotoxicity (MAETP inf)')
	('CML v4.8 2016', 'ecotoxicity: terrestrial', 'terrestrial ecotoxicity (TETP inf)')
	('CML v4.8 2016', 'energy resources: non-renewable', 'abiotic depletion potential (ADP): fossil fuels')
	('CML v4.8 2016', 'eutrophication', 'eutrophication (fate not incl.)')
	('CML v4.8 2016', 'human toxicity', 'human toxicity (HTP inf)')
	('CML v4.8 2016', 'material resources: metals/minerals', 'abiotic depletion potential (ADP): elements (ultimate reserves)')
	('CML v4.8 2016', 'ozone depletion', 'ozone layer depletion (ODP steady state)')
Use `list(this object)` to get the complete list.